---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [106]:
system("git submodule update --init --recursive")
# system("git submodule foreach --recursive git fetch && git submodule foreach --recursive && git reset --hard origin/main")
# Sys.setenv(PYTHONPATH = here::here("data-cleaning", "grouper"))


In [107]:
# Delete all R objects and run garbage collection so we start with a clean slate
rm(list = ls())
gc()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2574311,137.5,5040384,269.2,5040384,269.2
Vcells,4817537,36.8,21596529,164.8,20387082,155.6


In [108]:
# Load libraries and minor parameters
source(here::here("data-cleaning/r_scripts", "00_libraries-params.R"))


## Primary Parameters

In [109]:
# Prompt Options:
to_prompt <- FALSE # Whether to prompt for user inputs or not (if FALSE, default values in this cell will be used)
thai_prompt <- TRUE # Whether to prompt for thai grouper even if bypassing all other prompts

# IMPORTANT PARAMETERS:
full_claims_prefix <- "claims_extract_CLAIMS " # Include spaces if there are any
# Assign the correct file extension based on the year
# Read the contents of year_to_load.txt as a string
year_to_load <- fread(here::here("data-cleaning", "cache", "year_to_load.txt"), header = FALSE, colClasses = "character")[[1]]
print(year_to_load)
file_type <- if (year_to_load %in% c(2022:2023)) ".tsv" else ".csv"
print(file_type)
gcs_email <- "271591364028-compute@developer.gserviceaccount.com" # Service Account to use
gcp_proj <- system("gcloud config get-value project", intern = TRUE) # get current GCP Project
gcs_bucket <- "phic-claims-checkpoints" # Name of GCS bucket
gcs_pre_fpath <- "pre-tdrg" # Name of folder path prefix in GCS bucket for thai grouper input
gcs_post_fpath <- "post-tdrg" # Name of folder path prefix in GCS bucket for thai grouper output
gcs_spc_fpath <- "spc"
bq_dataset <- "phic" # bq dataset
bq_table <- paste0("temp_claims_", year_to_load) # temp bq table, later renamed to claims_20XX1231 in Push to BQ section

# Input:
to_sample <- TRUE # Whether to sample each split_part by sample_size_divisor (useful when iterating through code runs in quick succession)
sample_size_divisor <- 625 # Sample size divisor: Formula for sample size is total_rows / split_parts / sample_size_divisor. Choose between 5, 25, 125, and 625

# Output:
to_write <- TRUE # Whether to write out checkpoint_1 files (everything up until converting for grouper export)
to_combine <- TRUE # Whether to combine checkpoint 1 files into one data.table
to_group <- TRUE # Whether to export for the batch grouper or not
to_gcs <- TRUE # Whether to push to GCS or nt (Thai Grouper Input/Output)
to_bq <- TRUE # Whether to push to BQ or not
to_drop_bq <- TRUE # Whether to overwrite the existing BQ table

# Columns to drop
drop_cols <- c( # Which columns to drop
  paste0("ICDCODE", 13:14), # Start
  "ICCODED15", # note that ICDCODE15 is misspelled as ICCODED15 in all claims
  paste0("ICDCODE", 16:170) # Continuation
)

drop_cols_manual <- c(
  "MEMCAT_SUBCHILD_DESC" # Drop as per Cel's suggestion
)

# Flush files
to_flush_master <- FALSE # whether to flush aux-files, checkpoints, profvis, debug, cache, and samples
to_flush_partial <- FALSE # whether to flush partial files (raw files but split into split_parts parts)

# Control random behavior for reproducibility
global_seed <- seed <- 123 # Choose a number as seed
set.seed(seed) # Setting the seed reproducibility (Important for stuff like randomly choosing a pdx among multiple possible options)

# Machine Specifications
ram_size <- 64 # Input virtual or physical machine's RAM size here

# Print GCP project
message(paste("GCP Project:", gcp_proj, "\n"))


[1] "2022"
[1] ".tsv"


GCP Project: drg-pipeline 




In [110]:
# other parameters for manual adjustments
manual_patterns_to_replace <- c("\\b0800\\b", "\\b080\\b", "\\b0809\\b") # ICD codes to replace
manual_code_replacements <- c("O800", "O80", "O809") # ICD code replacements


## Secondary (Debug) Parameters

In [111]:
# Debug Parameters:
to_debug <- FALSE # whether to print debug statements
to_profvis <- FALSE # Conduct runtime duration analysis via profvis or not
to_view_checks <- TRUE # Whether to view checks and print statements
to_view_checks_parallel <- FALSE # Whether to view intermediate per split_part/chunk checks and print statements (not consolidated) when parallelized
to_parallel <- TRUE # Whether to parallelize each split_parts split_part into availableCores() chunks. Cuts down processing time from 120min to 15min.
to_split_read <- FALSE # WARNING: TRUE uses a lot of memory!!
to_dec_mem_usage <- TRUE # Whether to run rm() and gc() at every possible step
tmp_nrow <- Inf # Per split_part/chunk end_nrow (leave at Inf)
diff_chars <- 0
split_parts <- 15 # How many (integer) parts to split the 12+m row claims file into
end_nrow <- 10 # How many rows/entries to show in summary tables
max_bq_rows <- 15000 # Max rows to return for bq query
encode <- "unknown" # Choices: unknown, UTF-8, Latin-1
# sep <- "," # Choices: "," or "\t"
is_unix <- if (.Platform$OS.type == "unix") TRUE else FALSE # Detect operating system architecture

ram_buffer <- 0.1 # How much of a RAM buffer to leave for the OS


## File Paths

In [112]:
# Folder Path Prefixes:
clean_prefix <- "data-cleaning"
data_prefix <- file.path(clean_prefix, "data")
claims_prefix <- file.path(data_prefix, "claims")
checkpoint_1_prefix <- "checkpoint_1_claims_"
checkpoint_2_prefix <- "checkpoint_2_claims_"
checkpoint_3_prefix <- "DRG_Grouped_"
checkpoint_4_prefix <- "checkpoint_4_thai_grouper_input_"
checkpoint_5_prefix <- toupper(paste0(gcs_pre_fpath, "_", checkpoint_4_prefix))
checkpoint_6_prefix <- "checkpoint_6_grouped_claims"
checkpoint_7a_prefix <- "python_input_1"
checkpoint_7b_prefix <- "python_input_2"
checkpoint_10_prefix <- "stata"

# Folder Paths:
chkpt_path <- file.path(data_prefix, "checkpoints")
checkpoint_1_path <- file.path(chkpt_path, "checkpoint_1_partial_clean_claims")
checkpoint_2_path <- file.path(chkpt_path, "checkpoint_2_master_clean_claims")
checkpoint_3_path <- file.path(chkpt_path, "checkpoint_3_thai_partial_input")
checkpoint_4_path <- file.path(chkpt_path, "checkpoint_4_thai_master_input")
checkpoint_5_path <- file.path(chkpt_path, "checkpoint_5_thai_output")
checkpoint_6_path <- file.path(chkpt_path, "checkpoint_6_thai_merged")
checkpoint_7_path <- file.path(chkpt_path, "checkpoint_7_py_input")
checkpoint_8_path <- file.path(chkpt_path, "checkpoint_8_py_output")
checkpoint_9_path <- file.path(chkpt_path, "checkpoint_9_grouper_differences")
checkpoint_10_path <- file.path(chkpt_path, "checkpoint_10_stata")
cache_path <- file.path(clean_prefix, "cache")
aux_path <- file.path(data_prefix, "aux-files")
raw_claims_path <- file.path(claims_prefix, "raw")
raw_claims_parts_path <- file.path(claims_prefix, "raw", "parts")
raw_claims_samples_path <- file.path(claims_prefix, "raw", "samples")
profvis_path <- file.path(data_prefix, "profvis")
debug_path <- file.path("data-cleaning", "debug")

# File Paths
profvis_fpath <- here("data-cleaning", "data", "profvis", "profvis.html")

# Create directories:
created_dirs <- c() # Initialize empty vector
# For all "_path" variables, create a directory with that path
# Excludes "_fpath" variables
for (path in mget(ls(pattern = "_path$"), envir = .GlobalEnv)) {
  full_path <- here(path)
  if (!dir.exists(full_path)) {
    dir.create(full_path, recursive = TRUE)
    created_dirs <- c(created_dirs, full_path)
  }
}

# Print directories created if any
if (length(created_dirs) == 0) {
  message("All directories exist.\n")
} else {
  message("The following directories were created:\n")
  message(paste(paste(created_dirs, collapse = ",\n"), "\n"))
}

# Commonly Used File Paths:
full_claims_file <- here(
  raw_claims_path,
  paste0(full_claims_prefix, year_to_load, file_type) # Use the file_type variable here
)


All directories exist.




## Parameter Validation Logic

Checking if parameters are valid, especially for the current machine type (e.g. RAM size)

Additionally, ask for parameters if to_bypass_prompts is false

In [113]:
# Stop if forecasted memory usage is expected to crash the system
if (!split_parts == as.integer(split_parts) || split_parts <= 1) stop("ERROR: split_parts must be an integer greater than or equal to 2!")
if (ram_size <= 64 && split_parts <= 2) stop("Please set split_parts to at least 3 for 64 GB machines or it will likely crash")
if (ram_size <= 32 && split_parts <= 4) stop("Please set split_parts to at least 5 for 32 GB machines or it will likely crash")
if (ram_size <= 32 && to_split_read == TRUE) stop("Please set to_split_read to TRUE for 32 GB machines or it will likely crash")

# Function to prompt for input with default value
prompt_with_default <- function(prompt_text, default_value) {
  if (to_prompt) {
    user_input <- readline(prompt = paste0(prompt_text, " [Default: ", default_value, "]: "))
    if (user_input == "") {
      return(default_value)
    } else {
      return(user_input)
    }
  } else {
    message(
      paste0(
        "Using default value (",
        default_value, ") for ",
        deparse(substitute(default_value))
      )
    )
    return(default_value)
  }
}

# Set parameters based on prompts or defaults
full_claims_prefix <- prompt_with_default("Enter full_claims_prefix", full_claims_prefix)
full_claims_bq_prefix <- str_replace_all(full_claims_prefix, " ", "\\\\ ")
year_to_load <- prompt_with_default("Enter year_to_load", year_to_load)

# Prompt for whether to sample
to_sample <- as.logical(prompt_with_default("Sample data? (TRUE/FALSE)", to_sample))
sample_size_divisor <- as.integer(prompt_with_default("Enter sample_size_divisor", sample_size_divisor))

# Prompt for output options
to_write <- as.logical(prompt_with_default("Write output files? (TRUE/FALSE)", to_write))
to_combine <- as.logical(prompt_with_default("Combine files? (TRUE/FALSE)", to_combine))
to_group <- as.logical(prompt_with_default("Export for batch grouper? (TRUE/FALSE)", to_group))

# Flush options
to_flush_master <- as.logical(prompt_with_default("Flush master files? (TRUE/FALSE)", to_flush_master))
to_flush_partial <- as.logical(prompt_with_default("Flush partial files? (TRUE/FALSE)", to_flush_partial))

# RAM settings
ram_size <- as.numeric(prompt_with_default("Enter RAM size (GB)", ram_size))
# ram_buffer <- as.numeric(prompt_with_default("Enter RAM buffer (0.0-1.0)", ram_buffer))
ram_limit <- (1 - ram_buffer) * ram_size * (1024^3)

# Allowing each future_lapply session to use more memory
options(future.globals.maxSize = ram_limit)

# Compute the RAM limit for R processes, leaving the buffer for the OS
ram_limit_gb <- round((1 - ram_buffer) * ram_size, 0)

# Print the set RAM limit
message(sprintf("Setting future.globals.maxSize to: %.1f GB", ram_limit / (1024^3)))


Using default value (claims_extract_CLAIMS ) for full_claims_prefix

Using default value (2022) for year_to_load

Using default value (TRUE) for to_sample

Using default value (625) for sample_size_divisor

Using default value (TRUE) for to_write

Using default value (TRUE) for to_combine

Using default value (TRUE) for to_group

Using default value (FALSE) for to_flush_master

Using default value (FALSE) for to_flush_partial

Using default value (64) for ram_size

Setting future.globals.maxSize to: 57.6 GB



## Loading


### Load Required Libraries & Initial Functions

In [114]:
# Hide verbose outputs and warnings
options(verbose = FALSE) # Hide verbose output for script and library loading
options(warn = -1) # Hide warnings for script sourcing and library loading


In [115]:
scripts <- list( # List of scripts to source
  # lib_params = "00_libraries-params.R",
  cleaning = "01_cleaning-functions.R",
  clinical = "02_clinical-functions.R",
  timing_debug = "03_timing-debug-functions.R",
  summary = "04_summary-functions.R",
  io = "05_io-functions.R"
)

# Loop to source above scripts
for (script in scripts) source(here("data-cleaning/r_scripts", script))


In [116]:
# since verbose = TRUE is way too verbose compared to default
options(warn = 1) # Reenable warnings; see above comments


### Load Full Claims from GCS

In [117]:
# Load raw claims from GCS only if they don't exist on the VM yet
for (year in 2018:2023) {
  # Assign the correct file extension based on the year
  file_type <- if (year %in% c(2022:2023)) ".tsv" else ".csv"
  file_name <- paste0(full_claims_prefix, year, file_type)
  bq_name <- paste0(full_claims_bq_prefix, year, file_type)

  # Check if the file exists in the target directory
  file_path <- here(raw_claims_path, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    if (!is.null(gcp_proj) && gcp_proj == "drg-pipeline") {
      system(paste0("cd .. && gsutil cp gs://phic-claims-raw/", bq_name, " ", raw_claims_path),
        intern = FALSE, ignore.stderr = FALSE
      )
    } else {
      stop("Error: GCP Project is not null and is not drg-pipeline")
    }
  } else {
    message(paste("File", file_name, "already exists in the target directory. Skipping download.\n"))
  }
}


File claims_extract_CLAIMS 2018.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2019.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2020.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2021.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2022.tsv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2023.tsv already exists in the target directory. Skipping download.




### Load Mapping Data

#### Define Query Function for BigQuery

In [118]:
# Function to query BigQuery and optionally cache the results as a CSV
query_bq_to_dt <- function(query, file_path, cache, max_bq_rows) {
  if (file.exists(file_path)) {
    message(paste(basename(file_path), "already exists, loading from CSV"))
    dt <- fread(file_path)
  } else {
    message(paste("Querying BigQuery for", basename(file_path)))
    dt <- as.data.table(bq_table_download(
      bq_project_query(gcp_proj, query),
      n_max = max_bq_rows
    ))
    if (cache) {
      fwrite(dt, file_path)
    }
  }
  # return bq table as dt
  return(dt)
}


#### BigQuery Proper

In [119]:
# 1. Query and load `grouper_v5.proc`
proc_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".grouper_v5.proc` LIMIT ", max_bq_rows
)
proc <- query_bq_to_dt(proc_query, here(aux_path, "proc.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)
proc[, CODE := as.character(CODE)]

# 2. Query and load `phic.acr_rvs_map`
rvs_icd9_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".phic.acr_rvs_map` LIMIT ", max_bq_rows
)
rvs_icd9 <- query_bq_to_dt(rvs_icd9_query, here(aux_path, "rvs_icd9cm.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)

# Convert rvs to character and handle icd9cm conversion carefully
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]

# Merge with proc to classify by DRGUSE
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) &
  DRGUSE][!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

# 3. Query and load `phic.acr_procedure`
acr_rvs_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".phic.acr_procedure` LIMIT ", max_bq_rows
)
acr_rvs <- query_bq_to_dt(acr_rvs_query, here(aux_path, "acr_rvs.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)

# 4. Query and load `grouper_v5.i10`
i10_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".grouper_v5.i10` LIMIT ", max_bq_rows
)
tdrg_icd10 <- query_bq_to_dt(i10_query, here(aux_path, "i10.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)
setkey(tdrg_icd10, "CODE")

# Subset and assign to acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])

# 5. Query and load `icd.phl_icd10`
phl_icd10_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".icd.phl_icd10` LIMIT ", max_bq_rows
)
phl_icd10 <- query_bq_to_dt(phl_icd10_query, here(aux_path, "phl_icd10.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)

# Filter and process neoplasms
neoplasms_dt_actual <- as.data.table(phl_icd10[
  grepl("/", icd10),
  .(icd10)
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])

# 6. Query and load `drg-pipeline.grouper_v5.i10vx`
i10vx_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".grouper_v5.i10vx` LIMIT ", max_bq_rows
)
i10vx <- query_bq_to_dt(i10vx_query, here(aux_path, "i10vx.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)
setkey(i10vx, "code")
acc_icd <- unique(i10vx[, code])

# 7. Query and load `drg-pipeline.hci.temp_hci`
hci_query <- paste0(
  "SELECT * FROM `", gcp_proj,
  ".hci.temp_hci` LIMIT ", max_bq_rows
)
hci <- query_bq_to_dt(hci_query, here(aux_path, "hci.csv"),
  cache = FALSE, max_bq_rows = max_bq_rows
)


Querying BigQuery for proc.csv



Querying BigQuery for rvs_icd9cm.csv

Querying BigQuery for acr_rvs.csv

Querying BigQuery for i10.csv

Querying BigQuery for phl_icd10.csv

Querying BigQuery for i10vx.csv

Querying BigQuery for hci.csv



In [120]:
# cat(unique(hci$PMCC_NO), sep = "\n")
# cat(unique(hci$CAT_24), sep = "\n")


In [121]:
# Print the column names enclosed in quotes and separated by newlines
# colnames_list <- colnames(fread(full_claims_file, nrows = 1))
# cat(paste0('"', colnames_list, '"', collapse = "\n"))

# str(fread(full_claims_file, nrows = 10000, colClasses = "character"))


### Loading Cached Objects

Total Rows File, sample size computation, suffix definition

In [122]:
# Load cached total rows file if available, saves ~10 seconds of runtime
total_rows_file <- here(cache_path, paste0("total_rows_", year_to_load, ".rds"))
if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
  message(paste("Total Rows via cached object:", total_rows))
} else {
  total_rows <- fread(file = full_claims_file, select = 1L, header = TRUE, colClasses = "character")[, .N]
  saveRDS(total_rows, file = total_rows_file)
  message(paste("Total Rows via fread:", total_rows))
}

# Compute sample size when splitting and when not,
# only relevant when sampling
if (to_split) {
  sample_size <- ceiling(total_rows / split_parts / sample_size_divisor)
} else {
  sample_size <- ceiling(total_rows / sample_size_divisor)
}

# suffix appended to files to indicate if they are from sampled or full runs
suffix <- paste0(
  ifelse(to_sample, paste0("_sampled_", sample_size, "_"), "_full_")
)


Total Rows via cached object: 12757064



## Function Definitions

### Functions called in Process Chunk

Define Clean Data Function

In [123]:
# Define main clean data function, which does majority of the data cleaning on the claims file
clean_data <- function(dt) {
  # Rename columns based on column_mappings
  available_columns <- colnames(dt)

  # Convert source year to integer
  if ("ADMISSION_YEAR" %in% available_columns) {
    # do nothing
  } else {
    dt[, SRC_YR := as.integer(year_to_load)]
  }

  dt[, is_covid := FALSE]

  # Identify the columns to rename based on the mapping
  old_names <- available_columns[available_columns %in% names(column_mappings)]
  new_names <- sapply(old_names, function(col) column_mappings[[col]])

  # Rename the columns in the data.table
  setnames(dt, old = old_names, new = new_names)

  # Check if renaming was successful
  rename_success <- all(new_names %in% available_columns)

  dt[, c1 := clin_c1]
  dt[, c2 := clin_c2]

  # Convert time_adm and time_dis for 2022-2023 format

  # Strip fractional seconds and handle AM/PM conversion properly using as.POSIXct()
  dt[, time_adm := ifelse(grepl("AM|PM", time_adm),
    format(as.POSIXct(sub("\\.\\d+ ", " ", time_adm), format = "%m/%d/%Y %I:%M:%S %p"), "%H:%M"),
    time_adm
  )]

  dt[, time_dis := ifelse(grepl("AM|PM", time_dis),
    format(as.POSIXct(sub("\\.\\d+ ", " ", time_dis), format = "%m/%d/%Y %I:%M:%S %p"), "%H:%M"),
    time_dis
  )]

  # Collapse and clean ICD and RVS columns
  dt <- collapse_and_clean_icd_rvs(dt)

  # # Helper function to clean and compare clinical columns
  # clean_clinical_column <- function(col_name) {
  #   dt[, (paste0(col_name, "_orig")) := dt[[col_name]]]
  #   dt[, (col_name) := clean_column(
  #     dt[[col_name]], na_like_strings, neoplasms_dt_actual
  #   )]
  #   dt[, (paste0(col_name, "_orig")) := sapply(
  #     get(paste0(col_name, "_orig")), toString
  #   )]
  #   dt[, (col_name) := sapply(get(col_name), toString)]

  #   # Compare cleaning results
  #   dt[
  #     !is.na(get(paste0(col_name, "_orig"))) & get(col_name) !=
  #       get(paste0(col_name, "_orig")),
  #     .(
  #       old_code = get(paste0(col_name, "_orig")),
  #       new_code = get(col_name), count = .N
  #     ),
  #     by = .(get(paste0(col_name, "_orig")), get(col_name))
  #   ]
  # }

  # Helper function to clean and compare clinical columns
  clean_clinical_column <- function(col_name) {
    # Store the original column
    dt[, (paste0(col_name, "_orig")) := dt[[col_name]]]

    # Apply the clean_column function, which now returns a list of cleaned_col and is_covid
    cleaned_data <- clean_column(dt[[col_name]], na_like_strings, neoplasms_dt_actual)

    # Extract cleaned column and is_covid flag
    cleaned_col <- cleaned_data$cleaned_col
    is_covid_flag <- cleaned_data$is_covid

    # Update the cleaned column
    dt[, (col_name) := cleaned_col]

    # Store the original column as a string (for comparison purposes)
    dt[, (paste0(col_name, "_orig")) := sapply(get(paste0(col_name, "_orig")), toString)]

    # Convert the cleaned column to a string for comparison
    dt[, (col_name) := sapply(get(col_name), toString)]

    # Update is_covid column if needed
    # If is_covid is FALSE in dt, but the new is_covid_flag is TRUE, set is_covid to TRUE
    dt[, is_covid := ifelse(is_covid == FALSE & is_covid_flag == TRUE, TRUE, is_covid)]

    # Compare cleaning results (only rows where the cleaned column differs from the original)
    comparison <- dt[
      !is.na(get(paste0(col_name, "_orig"))) & get(col_name) != get(paste0(col_name, "_orig")),
      .(
        old_code = get(paste0(col_name, "_orig")),
        new_code = get(col_name), count = .N
      ),
      by = .(get(paste0(col_name, "_orig")), get(col_name))
    ]

    return(comparison)
  }

  # Clean and compare c1 and c2 columns
  c1_cleaning_comparison <- clean_clinical_column("c1")
  c2_cleaning_comparison <- clean_clinical_column("c2")

  # Function to replace multiple patterns with corresponding replacements
  replace_multiple_patterns <- function(text, patterns, replacements) {
    # Ensure patterns and replacements are the same length
    if (length(patterns) != length(replacements)) {
      stop("Patterns and replacements must have the same length.")
    }

    # Perform replacements
    modified_text <- stri_replace_all_regex(
      text,
      pattern = patterns,
      replacement = replacements,
      vectorize_all = FALSE # Apply all replacements simultaneously
    )

    # return the text after modification
    return(modified_text)
  }

  # Apply multi-replacement function to implement manual replacements
  dt[, clin_icd := lapply(clin_icd,
    replace_multiple_patterns,
    patterns = manual_patterns_to_replace,
    replacements = manual_code_replacements
  )]
  dt[, c1 := lapply(c1,
    replace_multiple_patterns,
    patterns = manual_patterns_to_replace,
    replacements = manual_code_replacements
  )]
  dt[, c2 := lapply(c2,
    replace_multiple_patterns,
    patterns = manual_patterns_to_replace,
    replacements = manual_code_replacements
  )]

  # Handle any lumped ICD codes by splitting them
  dt[, c1 := remove_lumped_icd_codes(c1)]
  # Handle any lumped ICD codes by splitting them
  dt[, c2 := remove_lumped_icd_codes(c2)]
  # Convert the cleaned columns into vectors
  dt[, c1 := split_to_vector(c1)]
  # Convert the cleaned columns into vectors
  dt[, c2 := split_to_vector(c2)]

  # # Remove lumped ICD codes
  # dt[, c1 := remove_lumped_icd_codes(c1)]
  # dt[, c2 := remove_lumped_icd_codes(c2)]
  # dt[, clin_rvs := remove_lumped_rvs_codes(clin_rvs)]

  # Clean clinical columns
  clean_clin_col_res <- clean_clinical_columns(dt)
  dt <- clean_clin_col_res$dt

  # Replace empty strings with NA and remap patient data
  replace_result <- replace_empty_with_na(dt = dt, to_view_checks)
  dt <- replace_result$return_data
  empty_strings_replaced_1 <- replace_result$return_replacement_summary

  remapping_results <- remap_patient_data(dt, to_view_checks)
  dt <- remapping_results$data
  return_summary_list <- list(
    rename_success = rename_success,
    ICD_replacements_1 = c1_cleaning_comparison,
    ICD_replacements_2 = c2_cleaning_comparison,
    pat_type_mapped = remapping_results$pat_type_mapped,
    pat_memcat_parent_mapped = remapping_results$pat_memcat_parent_mapped,
    pat_memcat_child_mapped = remapping_results$pat_memcat_child_mapped,
    clin_discharge_mapped = remapping_results$clin_discharge_mapped,
    claim_status_mapped = remapping_results$claim_status_mapped,
    pat_type_unmapped = remapping_results$pat_type_unmapped,
    memcat_parent_unmapped = remapping_results$memcat_parent_unmapped,
    memcat_child_unmapped = remapping_results$memcat_child_unmapped,
    discharge_unmapped = remapping_results$discharge_unmapped,
    claim_status_unmapped = remapping_results$claim_status_unmapped,
    discard_rvs_one = clean_clin_col_res$discard_rvs_one,
    discard_rvs_two = clean_clin_col_res$discard_rvs_two,
    empty_strings_replaced_1 = empty_strings_replaced_1
  )

  return(
    list(
      # data to return for further processing
      return_data = dt,
      # summary to return for checks and output
      return_summary = return_summary_list
    )
  )
}


Define RVS Mapping Function

In [124]:
# Define function to map RVS codes to ICD9 codes
map_rvs_icd9 <- function(clin_rvs, rvs_icd9) {
  split_codes <- split_rvs_codes(rvs_icd9)
  rvs_maps <- create_rvs_map_lists(split_codes$with_drg)

  rvs_map_solo_env <- as.environment(rvs_maps$rvs_map_solo)

  return(
    list(
      # main return variable (a column) to save back to dt
      icd9_list = get_icd9_codes(clin_rvs, rvs_map_solo_env),
      # other return variables that are for checks and outputs
      rvs_map_list = rvs_maps$rvs_map_list,
      rvss = unique(unlist(clin_rvs)),
      mappable_rvs = intersect(unique(unlist(clin_rvs)), rvs_icd9$rvs),
      unmappable_rvs = setdiff(unique(unlist(clin_rvs)), rvs_icd9$rvs),
      multi_mapped_rvs = intersect(unique(unlist(clin_rvs)), names(rvs_maps$rvs_map_list)),
      without_drg = unique(rvs_icd9[!rvs %in% names(rvs_maps$rvs_map_list)]$rvs)
    )
  )
}


Define ICD10 Application Function

In [125]:
implement_icd10_mapping <- function(c1, c2, clin_icd, tdrg_icd10) {
  # Step 1: Get all unique ICD codes from the provided columns (c1, c2, and clin_icd)
  icds <- get_unique_icd_codes(c1, c2, clin_icd)

  # Step 2: Create an environment for Thai ICD10 codes for faster lookup
  # This uses the unique set of ICD10 codes in the tdrg_icd10 table.
  thai_icd10_env <- create_thai_icd10_environment(
    unique(tdrg_icd10$CODE)
  )

  # Step 3: Create a second environment for Thai ICD10 neoplasm codes (those with slashes '/')
  neoplasms_env <- create_thai_icd10_environment(
    unique(tdrg_icd10[grepl("/", tdrg_icd10$CODE), "CODE"])
  )

  # Step 4: Find direct matches between the provided ICD codes (icds) and the Thai ICD10 environment
  direct_match_codes <- find_direct_icd_matches(
    icds, thai_icd10_env
  )

  # Step 5: Generate the full ICD10 mapping for the ICD codes,
  # considering both Thai ICD10 environment and neoplasms environment.
  icd_mapping_info <- generate_icd10_mapping(
    icds, thai_icd10_env, neoplasms_env
  )
  # Extract the mapping and the count of modified mappings
  icd_mapping <- icd_mapping_info$icd_mapping
  modified_count <- icd_mapping_info$modified_count

  # Step 6: Identify ICD codes that were not successfully mapped.
  unmatched_icds <- setdiff(icds, names(icd_mapping))

  # Step 7: If there are unmatched ICD codes, gather their source information (c1, c2, clin_icd)
  # and the count of occurrences in each column.
  if (length(unmatched_icds) > 0) {
    unmatched_sources <- data.table(
      code = unmatched_icds, source = NA_character_, count = 0
    )
    # Loop over the columns (c1, c2, clin_icd) to fill in source and count details for unmatched codes.
    for (col_name in c("c1", "c2", "clin_icd")) {
      col_values <- get(col_name)
      unmatched_sources[
        code %in% unlist(col_values),
        source := col_name
      ]
      unmatched_sources[
        code %in% unlist(col_values),
        count := count + table(unlist(col_values))[code]
      ]
    }
    # Order unmatched codes by their occurrence count in descending order
    unmatched_sources <- unmatched_sources[order(-count)]
  } else {
    # If there are no unmatched codes, return an empty data.table.
    unmatched_sources <- data.table()
  }

  # Step 8: Create a data.table containing the mapping between PHL (input) ICD10 codes
  # and Thai DRG ICD10 codes.
  icd10_map <- data.table(
    phl_icd10 = names(icd_mapping),
    tdrg_icd10 = unlist(icd_mapping)
  )

  # Step 9: If debugging is enabled, save the mapping to a CSV file in the cache directory.
  if (to_debug) {
    fwrite(icd10_map, paste0("cache/icd10_map_file_", year_to_load, ".csv"))
  }

  # Step 10: Create an environment from the ICD10 mapping for fast lookup during column mapping.
  icd10_env <- list2env(
    setNames(as.list(icd10_map$tdrg_icd10), icd10_map$phl_icd10)
  )

  # Step 11: Apply the ICD10 mapping to the columns c1, c2, and clin_icd
  # This updates these columns based on the generated ICD10 environment.
  mapped_columns <- apply_icd10_mapping_to_columns(
    c1, c2, clin_icd, icd10_env
  )

  # Step 12: Return a list containing the mapped columns and other information for further checks and outputs:
  # - The updated columns (c1, c2, clin_icd)
  # - The full ICD10 map (icd10_map_dt)
  # - The unique ICD codes
  # - Direct matches found
  # - Unmatched ICDs and their source information
  return(
    list(
      c1 = mapped_columns$c1,
      c2 = mapped_columns$c2,
      clin_icd = mapped_columns$clin_icd,
      icd10_map_dt = icd10_map,
      unique_icds = icds,
      direct_matches = direct_match_codes,
      unmatched = unmatched_icds,
      unmatched_sources = unmatched_sources
    )
  )
}


### Functions called in Main Logic Function

Define Process Chunk

In [126]:
##################################################################################################################################
################################################### START OF PROCESS CHUNK #######################################################
##################################################################################################################################

# Define process_chunk (not to be confused with process_part) that processes each part in nthreads chunks
process_chunk <- function(chunk, to_view_checks, rvs_icd9, tdrg_icd10, acc_pdx) {
  # Step 1: Set up environment for viewing or suppressing output
  # If 'to_view_checks' is TRUE, you can enable message viewing (commented out here).
  # Otherwise, sink (redirect output) to a temporary file to suppress output.
  if (to_view_checks) {
    # message("\rViewing checks")
  } else {
    sink(tempfile())
    on.exit(sink(), add = TRUE)
  }

  # Step 2: Clean the data in the 'chunk'
  clean_result <- clean_data(chunk)
  chunk <- clean_result$return_data # Update chunk with cleaned data
  if (to_debug) print("checkpoint 1") # Debug checkpoint
  if (to_debug) print(unique(chunk$c1)) # Print unique values in 'c1' for debugging

  # Step 3: Optionally save intermediate result to a file (debugging)
  if (to_debug) fwrite(chunk, "test1.csv")

  # Step 4: Map clinical RVS (Relative Value Scale) codes to ICD9 using 'rvs_icd9'
  rvs_mapping_result <- map_rvs_icd9(chunk$clin_rvs, rvs_icd9)
  # Store the mapped ICD9 list into the chunk
  chunk[, icd9_list := rvs_mapping_result$icd9_list]

  # Step 5: Optionally save another intermediate result to a file (debugging)
  if (to_debug) fwrite(chunk, "test2.csv")

  # Step 6: Extract columns c1, c2, and clin_icd for ICD10 mapping
  c1 <- chunk$c1
  c2 <- chunk$c2
  clin_icd <- chunk$clin_icd
  if (to_debug) print("checkpoint 2") # Debug checkpoint
  if (to_debug) print(unique(c1)) # Print unique values in 'c1' for debugging

  # Step 7: Perform ICD10 mapping using the extracted columns and 'tdrg_icd10' mapping data
  icd10_mapping_result <- implement_icd10_mapping(
    c1, c2, clin_icd, tdrg_icd10
  )
  # Update chunk with the mapped ICD10 codes
  chunk[, c1 := icd10_mapping_result$c1]
  chunk[, c2 := icd10_mapping_result$c2]
  chunk[, clin_icd := icd10_mapping_result$clin_icd]

  # Step 8: Replace any empty strings with NA values, returning a summary of replacements
  res2 <- replace_empty_with_na(dt = chunk, to_view_checks)
  chunk <- res2$return_data # Update chunk with cleaned data
  empty_strings_replaced_2 <- res2$return_replacement_summary # Store replacement summary

  if (to_debug) print("checkpoint 3") # Debug checkpoint
  if (to_debug) print(unique(chunk$c1)) # Print unique values in 'c1' for debugging

  # Step 9: Define a function to remove all whitespace from character vectors
  remove_whitespace <- function(x) {
    if (is.null(x) || length(x) == 0) {
      return(NA_character_) # Return NA for NULL or empty lists
    } else {
      return(gsub("\\s+", "", x)) # Remove all whitespace characters
    }
  }

  # Step 10: Apply the remove_whitespace function to the list columns 'c1', 'c2', and 'clin_icd'
  chunk[, c1 := lapply(c1, remove_whitespace)]
  chunk[, c2 := lapply(c2, remove_whitespace)]
  chunk[, clin_icd := lapply(clin_icd, remove_whitespace)]

  if (to_debug) print("checkpoint 4") # Debug checkpoint
  if (to_debug) print(unique(chunk$c1)) # Print unique values in 'c1' for debugging

  # Step 11: Optionally save a CSV file containing the column names and their types (for debugging)
  if (to_debug) fwrite(data.table(Column = colnames(chunk), Class = sapply(chunk, class)), "class.csv")

  # Step 12: Apply a function to find the primary diagnosis (pdx) based on 'c1', 'c2', and 'clin_icd'
  pdx_result <- apply_find_pdx(
    chunk$c1, chunk$c2, chunk$clin_icd, acc_pdx
  )
  # Store the primary diagnosis (pdx) and its code into the chunk
  chunk$pdx <- pdx_result$pdx
  chunk$pdx_code <- pdx_result$pdx_code

  # Step 13: Optionally save another intermediate result to a file (debugging)
  if (to_debug) fwrite(chunk, "test2c.csv")
  if (to_debug) print("checkpoint 5") # Debug checkpoint
  if (to_debug) print(unique(chunk$c1)) # Print unique values in 'c1' for debugging

  # Step 14: Define a function to remove the primary diagnosis (pdx) from list columns (c1, c2, clin_icd)
  remove_pdx_from_list <- function(pdx, lst) {
    if (!is.na(pdx)) {
      # Remove the primary diagnosis from the list
      lst <- setdiff(lst, pdx)
    }
    return(lst)
  }

  # Step 15: Apply the 'remove_pdx_from_list' function to each row of 'c1', 'c2', and 'clin_icd'
  chunk[, c1 := lapply(seq_len(.N), function(i) as.character(remove_pdx_from_list(pdx[i], c1[[i]])))]
  chunk[, c2 := lapply(seq_len(.N), function(i) as.character(remove_pdx_from_list(pdx[i], c2[[i]])))]
  chunk[, clin_icd := lapply(seq_len(.N), function(i) as.character(remove_pdx_from_list(pdx[i], clin_icd[[i]])))]

  if (to_debug) print("checkpoint 6") # Debug checkpoint
  if (to_debug) print(unique(chunk$c1)) # Print unique values in 'c1' for debugging

  # Step 16: Create a summary by combining clean results and ICD10 mapping information
  chunk_summary <- modifyList(
    clean_result$return_summary,
    list(
      unique_icds = icd10_mapping_result$unique_icds,
      direct_matches = icd10_mapping_result$direct_matches,
      unmatched = icd10_mapping_result$unmatched,
      unmatched_sources = icd10_mapping_result$unmatched_sources,
      icd10_map_dt = icd10_mapping_result$icd10_map_dt,
      rvss = rvs_mapping_result$rvss,
      mappable_rvs = rvs_mapping_result$mappable_rvs,
      unmappable_rvs = rvs_mapping_result$unmappable_rvs,
      multi_mapped_rvs = rvs_mapping_result$multi_mapped_rvs,
      without_drg = rvs_mapping_result$without_drg
    )
  )

  # Step 17: Optionally trigger garbage collection to reduce memory usage
  if (to_dec_mem_usage) gc() # Trigger garbage collection if needed

  # Step 18: Return the processed chunk and summary information
  return(
    list(
      return_chunk = chunk, # Return the processed chunk data
      return_summary = chunk_summary # Return the summary for checks and outputs
    )
  )
}

##################################################################################################################################
#################################################### END OF PROCESS CHUNK ########################################################
##################################################################################################################################


### Function called in Main Logic Section

Define a codeblock to avoid repeating it twice when to_profvis is TRUE and again if FALSE

Makes it easier to maintain as well, since we only need to modify one section instead of two

In [127]:
main_logic_func <- function() {
  # Start main execution logic
  ####################################################################################################################################
  ################################################## START OF SPLIT AND SAVE PART ####################################################
  ####################################################################################################################################

  separator <- if (file_type == ".tsv") "\t" else ","

  # Step 1: Read the header of the full claims file
  full_header <<- fread(
    file = full_claims_file,
    nrows = 1, colClasses = "character",
    header = TRUE, encoding = encode # , sep = separator
  )

  # Step 2: Check if the split part file already exists. If not, read the full claims file.
  if (!file.exists(here(raw_claims_parts_path, paste0(
    full_claims_prefix, year_to_load,
    "_part_", sprintf("%02d", split_parts),
    "_of_", split_parts, ".rds"
  )))) {
    # Read the full file into memory

    full_file <<- fread(
      file = full_claims_file, colClasses = "character",
      header = TRUE, encoding = encode # , sep = separator
    )
  }

  # Function to split the file into chunks and save them
  split_and_save <- function(split_and_save_part) {
    rows_per_part <- ceiling(total_rows / split_parts) # Calculate how many rows per part
    chunk_file <- here(raw_claims_parts_path, paste0(
      full_claims_prefix, year_to_load,
      "_part_", sprintf("%02d", split_and_save_part),
      "_of_", split_parts, ".rds"
    ))

    # Only process if the part does not already exist
    if (!file.exists(chunk_file)) {
      # Determine start and end rows for this chunk
      start_row <- (split_and_save_part - 1) * rows_per_part + 1
      end_row <- min(split_and_save_part * rows_per_part, total_rows)

      # Extract chunk of data for processing
      chunk_dt <- full_file[start_row:end_row]

      # Debug print the first 2 rows if in debug mode
      if (to_debug) print(head(chunk_dt), 2)

      # Save the chunk as an RDS file
      saveRDS(chunk_dt, chunk_file, compress = TRUE)

      # Optionally reduce memory usage
      if (to_dec_mem_usage) rm(chunk_dt)
      if (to_dec_mem_usage) gc()

      # Save processing time for this part
      split_processing_times[[split_loop_part]] <- as.numeric(
        difftime(Sys.time(), start_time, units = "secs")
      )

      # Print status update and estimate remaining time
      print_status_update(split_loop_part, split_parts, split_processing_times, "split")
    }
  }

  # Step 3: Split the file into parts and save them
  start_time <<- Sys.time() # Record start time
  for (split_loop_part in 1:split_parts) split_and_save(split_loop_part)

  ####################################################################################################################################
  ################################################## END OF SPLIT AND SAVE PART ######################################################
  ####################################################################################################################################

  # Step 4: Set up parallelization if required (Unix and non-Unix systems handled differently)
  if (to_parallel && !is_unix) plan(multisession, workers = nthreads)

  # Step 5: Loop through each part and process the partial files
  for (loop_part in 1:split_parts) {
    ##################################################################################################################################
    ################################################### START OF PROCESS PART ########################################################
    ##################################################################################################################################

    start_time <- Sys.time() # Record start time for processing
    partial_claims_file <<- here(raw_claims_parts_path, paste0(
      full_claims_prefix, year_to_load,
      "_part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
    ))

    # Step 6: Handle sampling logic if applicable
    if (to_sample) {
      sampled_claims_file <<- here(raw_claims_samples_path, paste0(
        "sampled_claims_", year_to_load, "_", sample_size,
        "_part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      ))
      ensure_sample_files_exist(loop_part) # Ensure sample files exist
    }

    # Step 7: Read the appropriate file (sample or full)
    read_result <- read_appropriate_file(loop_part, to_sample)
    read_in_dt <- read_result$read_result_dt # The data to process
    read_in_replacement_summary <- read_result$read_result_replacement_summary # Any replacements summary

    ##################################################################################################################################
    ############################################ START OF PARALLELIZE AND SUMMARIZE DATA #############################################
    ##################################################################################################################################

    # Step 8: Split the data into chunks for parallel processing
    chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
    chunks <- split(read_in_dt, rep(1:nthreads, each = chunk_size, length.out = nrow(read_in_dt)))

    # Step 9: Apply parallel processing (Unix uses 'mclapply', non-Unix uses 'future_lapply')
    if (to_parallel && is_unix) {
      if (to_debug) message("Conducting mclapply")
      parallel_results <- mclapply(
        chunks, process_chunk,
        mc.cores = nthreads,
        to_view_checks = to_view_checks,
        rvs_icd9 = rvs_icd9,
        tdrg_icd10 = tdrg_icd10,
        acc_pdx = acc_pdx
      )
    } else if (to_parallel && !is_unix) {
      if (to_debug) message("Conducting future_lapply")
      parallel_results <- future_lapply(
        chunks, process_chunk,
        to_view_checks = to_view_checks,
        rvs_icd9 = rvs_icd9,
        tdrg_icd10 = tdrg_icd10,
        acc_pdx = acc_pdx,
        future.seed = global_seed
      )
    } else {
      if (to_debug) message("Conducting lapply")
      parallel_results <- lapply(
        chunks, process_chunk,
        to_view_checks = to_view_checks,
        rvs_icd9 = rvs_icd9,
        tdrg_icd10 = tdrg_icd10,
        acc_pdx = acc_pdx
      )
    }

    # Step 10: Combine results from all parallel chunks
    parallel_summaries <- lapply(parallel_results, function(res) res$return_summary)
    rbound_dt <- rbindlist(lapply(parallel_results, function(res) res$return_chunk))

    combined_chunk_summary <- combine_chunk_summaries(
      parallel_summaries, tmp_nrow
    )

    # Step 11: Check for invalid primary diagnoses (PDx) and update the summary
    acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
    for (code in acc_pdx) {
      assign(code, TRUE, envir = acc_pdx_env)
    }
    invalid_pdx_indices <- which(
      !is.na(rbound_dt$pdx) & rbound_dt$pdx != "" &
        !sapply(rbound_dt$pdx, function(x) exists(x, acc_pdx_env))
    )
    if (length(invalid_pdx_indices) > 0) {
      message(paste("Invalid PDx found:", rbound_dt$pdx[invalid_pdx_indices]))
      combined_chunk_summary$pdx_success <- FALSE
    } else {
      combined_chunk_summary$pdx_success <- TRUE
    }

    if (to_dec_mem_usage) gc() # Reduce memory usage if necessary

    ##################################################################################################################################
    ############################################## END OF PARALLELIZE AND SUMMARIZE DATA #############################################
    ##################################################################################################################################

    summarized_dt <- rbound_dt # Store the summarized data
    combined_parallel_summary <- combined_chunk_summary # Store combined summary
    combined_parallel_summary$replacement_summary <- read_in_replacement_summary

    # Step 12: Write processed data to checkpoint file if required
    if (to_write) {
      saveRDS(
        summarized_dt, here(checkpoint_1_path, paste0(
          checkpoint_1_prefix, year_to_load, suffix,
          "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
        )),
        compress = TRUE
      )
    }

    # Step 13: Collect summaries for each part
    all_parts_summaries[[loop_part]] <- combined_parallel_summary
    processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(), start_time, units = "secs"))

    # Step 14: Update status and ETA
    print_status_update(loop_part, split_parts, processing_times, "clean")

    if (loop_part == 1) dim_dt <<- dim(summarized_dt)
    nrow_end[[loop_part]] <<- nrow(summarized_dt)

    # Step 15: Clean up memory after processing each part
    if (to_dec_mem_usage) {
      rm(read_in_dt, rbound_dt, summarized_dt)
      gc()
    }
  }

  # Step 16: Ensure that row counts match between parts
  for (nrow_part in 1:split_parts) {
    if (nrow_start[[nrow_part]] != nrow_end[[nrow_part]]) {
      warning(
        "WARNING: Row Count Mismatch! Part ", nrow_part,
        " has ", nrow_start[[nrow_part]], " starting rows and ",
        nrow_end[[nrow_part]], " ending rows\n"
      )
      stop("ERROR: Row Count Mismatch")
    }
  }
  message("\nRow Counts Match for All Parts\n")

  # Step 17: Combine all parts into a master data table if required
  if (to_combine) {
    for (read_part in 1:split_parts) {
      master_dt_list[[read_part]] <- readRDS(here(checkpoint_1_path, paste0(
        checkpoint_1_prefix, year_to_load, suffix,
        "part_", sprintf("%02d", read_part), "_of_", split_parts, ".rds"
      )))
    }
    master_dt <<- rbindlist(master_dt_list)

    if (to_debug) print(head(master_dt))
    if (to_dec_mem_usage) {
      if (to_group) rm(master_dt_list) else rm(master_dt_list)
      gc()
    }
    if (to_write) {
      saveRDS(master_dt, here(checkpoint_2_path, paste0(
        checkpoint_2_prefix, year_to_load, suffix, ".rds"
      )), compress = TRUE)
    }
  }

  # Step 18: Print final summaries
  print_summary_tables(
    combine_parts_summaries(all_parts_summaries, tmp_nrow),
    end_nrow
  )

  # Step 19: End parallelization session if on non-Unix system
  if (to_parallel && !is_unix) plan(sequential)

  # Return master data table for debugging purposes
  if (to_debug) {
    return(master_dt)
  }
}


## Main Processing Section

### Read, Process, and Export Data

Calls main_logic_func()

In [128]:
# Initialize Variables
all_parts_summaries <- master_dt_list <- list() # initialize lists
# if (to_group) master_grouper_input_list <- list()
dim_dt <- vector() # initialize vector for dt dimensions
processing_times <- split_processing_times <- nrow_start <- nrow_end <- numeric(split_parts)
master_dt <- data.table() # initialize data.tables
# if (to_group) master_grouper_input_dt <- data.table() # initialize data.tables
nthreads <- parallelly::availableCores() # detect available threads
message(paste0("Utilizing ", nthreads / 2, " cores (", nthreads, " threads)\n"))

# Call the main function with or without profvis
if (to_profvis) {
  saveWidget(profvis({
    main_logic_func()
  }), profvis_fpath)
} else {
  main_logic_func()
}


Utilizing 4 cores (8 threads)




Finished cleaning 15 of 15 parts in 37s (ETA 0s)        


Row Counts Match for All Parts





Rename Success:
 TRUE 

Table: ICD Normalized Text for clin c1 & c2 Before Splitting

|old_code     |new_code | diff_chars|
|:------------|:--------|----------:|
|B05.2+J17.1* |B052J171 |          4|
|I63.9+G46.7* |I639G467 |          4|
|B37.1+J17.2* |B371J172 |          4|
|I67.9+G46.8* |I679G468 |          4|
|E14.2+N08.3* |E142N083 |          4|
|A37.9+J17.0* |A379J170 |          4|
|A01.0+J17.0* |A010J170 |          4|
|E11.2+N08.3* |E112N083 |          4|
|B06.8+J17.1* |B068J171 |          4|
|D63.8*       |D638     |          2|


Table: Unique Before and After Mappings for Patient Type

|Original |Mapped |
|:--------|:------|
|DD       |D      |
|MM       |M      |


Table: Unique Before and After Mappings for Memcat Parent

|Original             |Mapped |
|:--------------------|:------|
|INDIRECT CONTRIBUTOR |I      |
|DIRECT CONTRIBUTOR   |D      |
|NA                   |NA     |


Table: Unique Before and After Mappings for Memcat Child

|Original                           

In [149]:
print(head(readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, ".rds"))), 100))


     id_year              id_series                           id_pin   date_adm
       <int>                 <char>                           <char>     <char>
  1:    2022 7009017009013012209007 12E7C6E880E651AB2AE91954A6753393 09/05/2022
  2:    2022 9103789103012902203019 481266642E0D8BB53DF37A196E87EDC1 07/11/2022
  3:    2022 5101955101060303201015 FE6730FFDE7474EAD38F330A6C4B403F 10/03/2022
  4:    2022 6101306101130203211016 565711CAE81EC5F7DB4B3B031478F47E 12/14/2022
  5:    2022 9003479003022802203009 21BEAAB1E4AD4841C85914954A97EB71 07/28/2022
  6:    2022 5004065004002212204005 4F2CA837C4E306571C8B5DA972AF273E 10/19/2022
  7:    2022 5008465008103103218005 A15CB8950194BB4C220ACA2D061CE0D1 12/17/2022
  8:    2022 4106334106031902206014 5632C6CF96719B208D7479DFB502D968 07/07/2022
  9:    2022 6000966000212602220006 0B0FC287DC77C68884E6BBFFA10192CE 04/22/2022
 10:    2022 0205910205140103215020 522EE8613C35CF6E141FDE5D95365490 11/18/2022
 11:    2022 8102378102072802202018 A0A8

In [130]:
if (exists("master_dt")) {
  result <- data.table::copy(master_dt)
  rm(master_dt)
  gc()
} else {
  result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, ".rds")))
  gc()
}

# Filter rows where any element in c1 contains "c("
if (to_debug) filtered_rows <- as.data.table(result[sapply(c1, function(x) any(grepl("c\\(", x)))])

# Display the structure of the filtered rows
if (to_debug) str(filtered_rows)

# Track invalid age corrections
invalid_age_before <- nrow(result[pat_age < -1 | pat_age > 124, .(id_series)])

pat_age_less_than_or_equal_to_neg_one <- result[pat_age <= -1, .(id_series, pat_age, c1, c2)]
fwrite(pat_age_less_than_or_equal_to_neg_one, here(debug_path, "pat_age_less_than_or_equal_to_neg_one.csv"))

pat_age_between_zero_and_neg_one <- result[pat_age < 0 & pat_age > -1, .(id_series, pat_age, c1, c2)]
fwrite(pat_age_between_zero_and_neg_one, here(debug_path, "pat_age_between_zero_and_neg_one.csv"))

# Use c1_orig and c2_orig as clin_c1 and clin_c2
result[, clin_c1 := c1_orig]
result[, clin_c2 := c2_orig]
result[, c("c1_orig", "c2_orig") := NULL]

# Use icd9_list as clin_proc
result[, clin_proc := icd9_list]
result[, icd9_list := NULL]

# Remove the primary diagnosis from the list of secondary diagnoses
result[, clin_icd := Map(function(pdx_var, sdx_var) sdx_var[sdx_var != pdx_var], pdx, clin_icd)]
result[, clin_sdx := clin_icd]
result[, clin_icd := NULL]

# Ensure that NA check respects the structure of c1 and returns a logical vector of the same length as result
if (to_debug) str(result[sapply(c1, function(x) length(x) > 1 && !all(is.na(unlist(x))))])

if (!"pat_bdate" %in% colnames(dt)) {
  result[, pat_bdate := NA_Date_]
}
date_cols <- c("date_adm", "date_dis", "date_rec", "date_ref", "date_check", "pat_bdate", "date_ext")
# Function to convert and replace dates before 1900-01-01 with NA
convert_and_filter_dates <- function(x) {
  converted_dates <- as.Date(x, format = "%m/%d/%Y")
  # Replace dates before 1900-01-01 with NA
  converted_dates[converted_dates < as.Date("1900-01-01")] <- NA_Date_
  return(converted_dates)
}

if (is_unix) {
  result[, (date_cols) := mclapply(.SD, convert_and_filter_dates, mc.cores = parallel::detectCores()), .SDcols = date_cols]
} else {
  result[, (date_cols) := lapply(.SD, convert_and_filter_dates), .SDcols = date_cols]
}

# Process time columns
time_cols <- c("time_adm", "time_dis")
standardize_time <- function(x) {
  x <- ifelse(is.na(x), "00:00:00", paste0(x, ":00"))
  as.ITime(x)
}
if (is_unix) {
  result[, (time_cols) := mclapply(.SD, standardize_time, mc.cores = parallel::detectCores()), .SDcols = time_cols]
} else {
  result[, (time_cols) := lapply(.SD, standardize_time), .SDcols = time_cols]
}

# Convert ITime object to character in HH:MM:SS format
result[, time_adm := strftime(time_adm, format = "%H:%M:%S")]
result[, time_dis := strftime(time_dis, format = "%H:%M:%S")]

# Convert date_adm and date_dis from Asia/Manila to UTC
result[, date_adm := as.POSIXct(paste(date_adm, time_adm), format = "%Y-%m-%d %H:%M:%S", tz = "Asia/Manila")]
result[, date_dis := as.POSIXct(paste(date_dis, time_dis), format = "%Y-%m-%d %H:%M:%S", tz = "Asia/Manila")]

# Process logical columns
result[, clin_outpatient := as.logical(as.integer(clin_outpatient))]
result[, clin_emergency := as.logical(as.integer(clin_emergency))]

# Process numeric columns
if (!"pat_bwt" %in% colnames(dt)) {
  result[, pat_bwt := NA_real_]
}
num_cols <- c("pat_age", "pat_bwt", "clin_discharge", "claim_payout", "claim_charge", "id_year", "pdx_code")
if (is_unix) {
  result[, (num_cols) := mclapply(.SD, as.numeric, mc.cores = parallel::detectCores()), .SDcols = num_cols]
} else {
  result[, (num_cols) := lapply(.SD, as.numeric), .SDcols = num_cols]
}

# Process integer columns
int_cols <- c("clin_discharge", "id_year", "pdx_code")
if (is_unix) {
  result[, (int_cols) := mclapply(.SD, as.integer, mc.cores = parallel::detectCores()), .SDcols = int_cols]
} else {
  result[, (int_cols) := lapply(.SD, as.integer), .SDcols = int_cols]
}

# Process character columns
char_cols <- c("id_hcp", "pat_type", "clin_acc", "pat_rel", "pat_sex", "pat_memcat_parent", "pat_memcat_child", "claim_status", "pdx")
if (is_unix) {
  result[, (char_cols) := mclapply(.SD, as.character, mc.cores = parallel::detectCores()), .SDcols = char_cols]
} else {
  result[, (char_cols) := lapply(.SD, as.character), .SDcols = char_cols]
}

# Age correction logic
invalid_ages_before_correction <- result[pat_age < 0 | pat_age > 124, .N]
invalid_age_ids_before <- result[pat_age < 0 | pat_age > 124, id_series]

result[pat_age < 0 & pat_age >= -1, pat_age := 0]
result[pat_age < -1 & is.na(pat_bdate), pat_age := NA_integer_]
result[pat_age > 124, pat_age := NA_integer_]
result[pat_age < -1 & !is.na(pat_bdate), pat_age := floor(as.numeric(interval(pat_bdate, date_adm) / years(1)))]

# Regenerate or correct DOB
invalid_bdate_before <- result[is.na(pat_bdate), .N]
invalid_bdate_ids_before <- result[is.na(pat_bdate), id_series]

result[!is.na(pat_age), pat_bdate := dmy(generate_dob(format(pat_bdate, "%Y-%m-%d"), pat_age, format(date_adm, "%Y-%m-%d")))]
result[!is.na(pat_bdate) & pat_bdate <= date_adm, pat_age := floor(as.numeric(interval(pat_bdate, date_adm) / years(1)))]
result[pat_age < -1 & !is.na(pat_bdate) & pat_bdate > date_adm, pat_age := NA_integer_]
result[pat_age > 124 | pat_age < 0, pat_age := NA_integer_]

# Save invalid age rows to CSV
invalid_age_path <- here("data-cleaning", "debug", "invalid_age.csv")
fwrite(data.table(id_series = invalid_age_ids_before), invalid_age_path)

# Save invalid birthdate rows to CSV
invalid_bdate_path <- here("data-cleaning", "debug", "invalid_bdate.csv")
fwrite(data.table(id_series = invalid_bdate_ids_before), invalid_bdate_path)

# Print messages for invalid ages corrected
invalid_ages_after_correction <- result[pat_age < 0 | pat_age > 124, .N]
message(
  "Number of invalid ages corrected: ", invalid_ages_before_correction - invalid_ages_after_correction,
  ". Invalid ages are those with a value less than 0 or greater than 124, which were reset to NA or corrected."
)

# Print messages for invalid birthdates corrected
invalid_bdate_after <- result[is.na(pat_bdate), .N]
message(
  "Number of invalid birthdates corrected: ", invalid_bdate_before - invalid_bdate_after,
  ". Invalid birthdates are missing values (NA), which were corrected based on age and admission dates."
)

result[, pat_ageday := NA_integer_]

# Process pat_ageday for patients younger than 1 year
invalid_ageday_before <- result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 & is.na(pat_ageday), .N]
invalid_ageday_ids_before <- result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 & is.na(pat_ageday), id_series]

result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 & is.na(pat_ageday), pat_ageday := 3]

result[
  !is.na(pat_age) & pat_age >= 0 & pat_age < 1 & !is.na(date_adm) & !is.na(pat_bdate),
  pat_ageday := as.integer(difftime(date_adm, pat_bdate, units = "days"))
]

if ("ageday" %in% colnames(result)) {
  result[, ageday := NULL]
}
gc()
# Print messages for invalid ageday corrections
invalid_ageday_after <- result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 & is.na(pat_ageday), .N]
message(
  "Number of agedays generated: ", invalid_ageday_before - invalid_ageday_after,
  ". Agedays generated are for where the patient is younger than 1 year, so the exact number of days was generated."
)

# Assuming acc_icd_env is an environment containing acc_icd codes
acc_icd_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_icd) {
  assign(code, TRUE, envir = acc_icd_env)
}

# Modify the data.table operation to use mget with the acc_icd_env
result[, clin_sdx := lapply(clin_sdx, function(row) {
  codes <- unlist(row)
  valid_codes <- codes[!is.na(mget(codes, envir = acc_icd_env, ifnotfound = NA_character_))]
  if (length(valid_codes) > 0) {
    return(valid_codes)
  } else {
    return(NA_character_)
  }
})]

# Optionally unlist each element of clin_sdx
result[, clin_sdx := lapply(clin_sdx, unlist)]

na_replaced_result <- replace_empty_with_na(result)

result <- na_replaced_result$return_data

# Process character columns and convert to UTF-8
if (is_unix) {
  result[, (char_cols) := mclapply(.SD, function(col) iconv(col, from = "", to = "UTF-8"), mc.cores = parallel::detectCores()), .SDcols = char_cols]
} else {
  result[, (char_cols) := lapply(.SD, function(col) iconv(col, from = "", to = "UTF-8")), .SDcols = char_cols]
}

# Function to clean data by handling missing values and replacing invalid entries
clean_data_columns <- function(data) {
  # Identify and process character columns
  char_cols <- names(data)[sapply(data, is.character)]

  # If running on Unix, apply parallel processing for character columns
  if (is_unix) {
    data[, (char_cols) := mclapply(.SD, function(col) {
      # Replace "None" and empty strings with NA in character columns
      col[col %in% c("None", "")] <- NA_character_
      return(col) # Return modified column
    }, mc.cores = parallel::detectCores()), .SDcols = char_cols]
  } else {
    # Apply sequential processing for character columns (non-Unix systems)
    data[, (char_cols) := lapply(.SD, function(col) {
      col[col %in% c("None", "")] <- NA_character_
      return(col) # Return modified column
    }), .SDcols = char_cols]
  }

  # Identify and process numeric columns
  num_cols <- names(data)[sapply(data, is.numeric)]

  # If running on Unix, apply parallel processing for numeric columns
  if (is_unix) {
    data[, (num_cols) := mclapply(.SD, function(col) {
      # Replace NaN values with NA in numeric columns
      col[is.nan(col)] <- NA_real_
      return(col) # Return modified column
    }, mc.cores = parallel::detectCores()), .SDcols = num_cols]
  } else {
    # Apply sequential processing for numeric columns (non-Unix systems)
    data[, (num_cols) := lapply(.SD, function(col) {
      col[is.nan(col)] <- NA_real_
      return(col) # Return modified column
    }), .SDcols = num_cols]
  }

  # Identify and process list columns
  list_cols <- names(data)[sapply(data, is.list)]

  # If running on Unix, apply parallel processing for list columns
  if (is_unix) {
    data[, (list_cols) := mclapply(.SD, function(col) {
      # For each list element, check if it contains characters and replace "None" and empty strings with NA
      lapply(col, function(x) {
        if (is.character(x)) x[x %in% c("None", "")] <- NA_character_
        return(x) # Return modified list element
      })
    }, mc.cores = parallel::detectCores()), .SDcols = list_cols]
  } else {
    # Apply sequential processing for list columns (non-Unix systems)
    data[, (list_cols) := lapply(.SD, function(col) {
      lapply(col, function(x) {
        if (is.character(x)) x[x %in% c("None", "")] <- NA_character_
        return(x) # Return modified list element
      })
    }), .SDcols = list_cols]
  }

  return(data) # Return the cleaned data.table
}

# Apply the cleaning function to the result data.table
result <- clean_data_columns(result)

# Convert string columns to arrays, handling different delimiters: comma, comma with space, single pipe, and double pipe
array_columns <- c("id_hcp")
split_pattern <- "\\s*,\\s*|\\|\\||\\|" # Regex pattern to handle commas, single pipes, and double pipes

if (is_unix) {
  result[, (array_columns) := mclapply(.SD, function(x) {
    # Split based on the specified pattern (comma, comma with space, single pipe, or double pipe)
    x <- strsplit(x, split_pattern)
    # Handle empty or NA entries
    lapply(x, function(y) if (length(y) == 0L || all(is.na(y))) character(0) else y)
  }, mc.cores = parallel::detectCores()), .SDcols = array_columns]
} else {
  result[, (array_columns) := lapply(.SD, function(x) {
    # Split based on the specified pattern (comma, comma with space, single pipe, or double pipe)
    x <- strsplit(x, split_pattern)
    # Handle empty or NA entries
    lapply(x, function(y) if (length(y) == 0L || all(is.na(y))) character(0) else y)
  }), .SDcols = array_columns]
}

# Ensure 'clin_sdx', 'clin_proc', and 'id_hcp' are not NULL
list_columns <- c("clin_sdx", "clin_proc", "id_hcp")
if (is_unix) {
  result[, (list_columns) := mclapply(.SD, function(col) {
    lapply(col, function(x) if (is.null(x) || length(x) == 0L || all(is.na(x))) character(0) else x)
  }, mc.cores = parallel::detectCores()), .SDcols = list_columns]
} else {
  result[, (list_columns) := lapply(.SD, function(col) {
    lapply(col, function(x) if (is.null(x) || length(x) == 0L || all(is.na(x))) character(0) else x)
  }), .SDcols = list_columns]
}

setnames(result, c("pdx", "pdx_code"), c("clin_pdx", "clin_pdx_source"))

setcolorder(result, c(
  "id_year", "id_series", "id_pin", "id_hci", "id_hcp", "date_adm", "time_adm", "date_dis", "time_dis",
  "date_rec", "date_ref", "date_check", "date_ext", "pat_type", "pat_rel", "pat_bdate", "pat_age",
  "pat_ageday", "pat_sex", "pat_bwt", "pat_memcat_parent", "pat_memcat_child", "is_covid", "claim_status", "claim_payout",
  "claim_charge", "clin_discharge", "clin_outpatient", "clin_emergency", "clin_acc", "clin_c1", "c1", "clin_c2", "c2",
  "clin_sdx", "clin_proc", "clin_rvs", "clin_pdx", "clin_pdx_source"
))


# Use a temporary column to avoid self-reference
result[, temp_clin_discharge := as.integer(clin_discharge)]

# Assign the temp column back to clin_discharge
result[, clin_discharge := temp_clin_discharge]

# Remove the temporary column
result[, temp_clin_discharge := NULL]

# # Apply format_id function to each column in parallel or sequentially
# columns_to_format <- c("id_series", "id_pin", "id_hci")
# # Define the format_id function that handles both numerical and alphanumeric formats
# format_id <- function(x) {
#   x <- as.character(x) # Ensure the data is in character format
#   numeric_x <- suppressWarnings(as.numeric(x)) # Try to convert to numeric
#   # Handle numeric values
#   is_numeric <- !is.na(numeric_x) # Identify numeric values
#   x[is_numeric] <- trimws(formatC(numeric_x[is_numeric], format = "f", digits = 0)) # Format numeric values without decimals
#   # Handle non-numeric values (leave as they are)
#   x[!is_numeric] <- trimws(x[!is_numeric])
#   # Return the cleaned-up values
#   return(x)
# }
# # Apply format_id function to each column safely, either in parallel (for Unix) or sequentially (for non-Unix)
# if (is_unix) {
#   # Make a copy of the columns to avoid directly accessing the data.table object in parallel
#   formatted_cols <- mclapply(columns_to_format, function(col) {
#     column_data <- result[[col]] # Extract column data outside the parallel loop
#     return(format_id(column_data)) # Apply format_id to the column
#   }, mc.cores = parallel::detectCores())
# } else {
#   formatted_cols <- lapply(columns_to_format, function(col) {
#     column_data <- result[[col]] # Extract column data
#     return(format_id(column_data)) # Apply format_id to the column
#   })
# }
# # Assign the formatted results back to the respective columns
# for (i in seq_along(columns_to_format)) {
#   result[[columns_to_format[i]]] <- formatted_cols[[i]]
# }

bw_dist <- c(
  runif(2, 0.5, 0.9), # Random birthweight between 0.5 and 0.9 for 2 newborns
  runif(8, 1.1, 1.4), # Random birthweight between 1.1 and 1.4 for 8 newborns
  runif(19, 1.6, 1.9), # Random birthweight between 1.6 and 1.9 for 19 newborns
  runif(95, 2.1, 2.4), # Random birthweight between 2.1 and 2.4 for 95 newborns
  runif(381, 2.6, 2.9), # Random birthweight between 2.6 and 2.9 for 381 newborns
  runif(375, 3.1, 3.4), # Random birthweight between 3.1 and 3.4 for 375 newborns
  runif(115, 3.5, 4.0), # Random birthweight between 3.5 and 4.0 for 115 newborns
  runif(6, 0.5, 4.0) # Random birthweight between 0.5 and 4.0 for 6 newborns
)

# Create the zero_mask condition where pat_age is between 0 and 1 (newborns)
zero_mask <- result[, pat_age >= 0 & pat_age < 1]

# Apply birthweight only if pat_bwt is NA and zero_mask is TRUE
result[(pat_bwt < 0 | is.na(pat_bwt)) & zero_mask, pat_bwt := sapply(.SD$pat_bwt, function(x) sample(bw_dist, 1)), .SDcols = "pat_bwt"]

saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")), compress = TRUE)


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2796592,149.4,5040384,269.2,5040384,269.2
Vcells,7522664,57.4,21596529,164.8,20387082,155.6


Number of invalid ages corrected: 6. Invalid ages are those with a value less than 0 or greater than 124, which were reset to NA or corrected.

Number of invalid birthdates corrected: 20415. Invalid birthdates are missing values (NA), which were corrected based on age and admission dates.



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2794465,149.3,5040384,269.2,5040384,269.2
Vcells,6768847,51.7,21596529,164.8,21579085,164.7


Number of agedays generated: 2263. Agedays generated are for where the patient is younger than 1 year, so the exact number of days was generated.



In [131]:
print(head(result))


   id_year              id_series                           id_pin id_hci
     <num>                 <char>                           <char> <char>
1:    2022 7009017009013012209007 12E7C6E880E651AB2AE91954A6753393 440901
2:    2022 9103789103012902203019 481266642E0D8BB53DF37A196E87EDC1 Z03199
3:    2022 5101955101060303201015 FE6730FFDE7474EAD38F330A6C4B403F Z04606
4:    2022 6101306101130203211016 565711CAE81EC5F7DB4B3B031478F47E 591201
5:    2022 9003479003022802203009 21BEAAB1E4AD4841C85914954A97EB71 300806
6:    2022 5004065004002212204005 4F2CA837C4E306571C8B5DA972AF273E D00060
   id_hcp            date_adm time_adm            date_dis time_dis   date_rec
   <list>              <POSc>   <char>              <POSc>   <char>     <Date>
1:  34333 2022-09-05 11:30:00 11:30:00 2022-09-06 16:17:00 16:17:00 2022-10-31
2:  59359 2022-07-11 17:02:00 17:02:00 2022-07-13 15:25:00 15:25:00 2022-09-21
3:  59139 2022-10-03 13:26:00 13:26:00 2022-10-03 13:26:00 13:26:00 2023-03-06
4:  38465 202

In [132]:
# Convert both id_hci and PMCC_NO to characters to ensure consistency for joining
result[, id_hci := as.character(id_hci)]
hci[, PMCC_NO := as.character(PMCC_NO)]

# Remove leading zeros from numeric PMCC_NO values, while preserving non-numeric values
hci[, PMCC_NO_stripped := fifelse(
  grepl("^[0-9]+$", PMCC_NO),
  sub("^0+", "", PMCC_NO), # Remove leading zeros from numeric strings
  PMCC_NO # Keep non-numeric values unchanged
)]

# Keep only necessary columns from hci for the join
hci_subset <- hci[, .(PMCC_NO_stripped, SOC_SECTOR, INST_NAME, CAT_24, REGION_NAME, PROVINCE_NAME)]

# Perform a left join, keeping all rows in result and only matching rows from hci
result <- merge(
  result,
  hci_subset,
  by.x = "id_hci",
  by.y = "PMCC_NO_stripped",
  all.x = TRUE, # Keep all rows from result
  all.y = FALSE # Only include matching rows from hci
)

# ----- Filter rows by allowed CAT_24 categories -----
allowed_categories <- c("INFIRMARY/DISPENSARY", "LEVEL 1 HOSPITAL", "LEVEL 2 HOSPITAL", "LEVEL 3 HOSPITAL")

# Count rows where CAT_24 is not in allowed categories
count_excluded <- result[!(CAT_24 %in% allowed_categories), .N]
cat("Number of rows where CAT_24 is not in the allowed categories:", count_excluded, "\n")

# Print rows to be excluded
if (to_debug) cat("Rows to be dropped due to invalid CAT_24:\n")
if (to_debug) print(result[!(CAT_24 %in% allowed_categories)])

# Drop rows where CAT_24 is not in the allowed categories
result <- result[CAT_24 %in% allowed_categories]
if (to_debug) str(result)
# ----- Filter rows where clin_outpatient is TRUE -----
# Count rows where clin_outpatient is TRUE
count_clin_outpatient_true <- result[clin_outpatient == TRUE, .N]
cat("Number of rows where clin_outpatient is TRUE:", count_clin_outpatient_true, "\n")

# Print rows to be excluded
if (to_debug) cat("Rows to be dropped due to clin_outpatient being TRUE:\n")
if (to_debug) print(result[clin_outpatient == TRUE])

# Drop rows where clin_outpatient is TRUE
result <- result[clin_outpatient != TRUE]
if (to_debug) str(result)
# ----- Filter rows where claim_status is not "G" -----
# Count rows where claim_status is not "G"
count_claim_status_not_G <- result[claim_status != "G", .N]
cat("Number of rows where claim_status is not 'G':", count_claim_status_not_G, "\n")

# Print rows to be excluded
if (to_debug) cat("Rows to be dropped due to claim_status not being 'G':\n")
if (to_debug) print(result[claim_status != "G"])

# Drop rows where claim_status is not "G"
result <- result[claim_status == "G"]
if (to_debug) str(result)
# Convert 'c1' from a list of character vectors into a single concatenated string
result[, c1_spc := sapply(c1, function(x) {
  if (is.null(x) || all(is.na(x))) {
    return(NA_character_) # Return NA if the list is empty or all values are NA
  } else {
    return(paste(sort(unique(x)), collapse = ",")) # Sort, remove duplicates, and concatenate
  }
})]

# ----- Handle duplicate rows based on specific columns -----
duplicate_columns <- c("id_pin", "pat_type", "pat_age", "pat_sex", "date_adm", "date_dis", "c1_spc", "claim_payout")

# Count duplicate rows based on the specified columns
count_duplicates <- result[duplicated(result[, ..duplicate_columns]), .N]
cat("Number of duplicated rows based on specified columns:", count_duplicates, "\n")

# Print duplicate rows
if (to_debug) cat("Rows with duplicated combinations of the specified columns:\n")
if (to_debug) print(result[duplicated(result[, ..duplicate_columns])])

# Drop duplicate rows, keeping only the first occurrence
result <- result[!duplicated(result[, ..duplicate_columns])]
if (to_debug) str(result)
# ----- Remove unnecessary columns and reorder remaining columns -----
# Remove unnecessary columns
result[, c("c1", "c1_spc", "c2", "clin_rvs") := NULL]

# Perform garbage collection to free up memory
gc()

# Set the column order to a specified structure
setcolorder(result, c(
  "id_year", "id_series", "id_pin", "id_hci", "id_hcp", "date_adm", "time_adm", "date_dis", "time_dis",
  "date_rec", "date_ref", "date_check", "date_ext", "pat_type", "pat_rel", "pat_bdate", "pat_age",
  "pat_ageday", "pat_sex", "pat_bwt", "pat_memcat_parent", "pat_memcat_child", "is_covid", "claim_status", "claim_payout",
  "claim_charge", "clin_discharge", "clin_outpatient", "clin_emergency", "clin_acc", "clin_c1", "clin_c2",
  "clin_sdx", "clin_proc", "clin_pdx", "clin_pdx_source", "SOC_SECTOR", "CAT_24", "INST_NAME", "REGION_NAME", "PROVINCE_NAME"
))


Number of rows where CAT_24 is not in the allowed categories: 5128 
Number of rows where clin_outpatient is TRUE: 7631 
Number of rows where claim_status is not 'G': 344 


Number of duplicated rows based on specified columns: 0 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2828798,151.1,5040384,269.2,5040384,269.2
Vcells,7145761,54.6,21596529,164.8,21579085,164.7


In [133]:
# str(result)


In [134]:
saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")), compress = TRUE)


Check if any of our identifiers are in exponential form e.g. "1.9e+07"

In [135]:
# str(result_after_cleaning)
print(result[grepl("e", id_series)])
print(result[grepl("e", id_pin)])
print(result[grepl("e", id_hci)])


Key: <id_hci>
Empty data.table (0 rows and 41 cols): id_year,id_series,id_pin,id_hci,id_hcp,date_adm...
Key: <id_hci>
Empty data.table (0 rows and 41 cols): id_year,id_series,id_pin,id_hci,id_hcp,date_adm...
Key: <id_hci>
Empty data.table (0 rows and 41 cols): id_year,id_series,id_pin,id_hci,id_hcp,date_adm...


In [136]:
# str(result)
# fwrite(result, "test.csv")
# Search for rows where any element in clin_sdx is "A"
result[, if (any(sapply(clin_sdx, function(row) "A" %in% row))) print(.SD), by = 1:nrow(result)]


nrow
<int>


In [137]:
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))


In [138]:
# date_cols <- c("date_adm", "date_dis", "date_rec", "date_ref", "date_check", "pat_bdate", "date_ext")

# # Function to convert and replace dates before 1900-01-01 with NA
# convert_and_filter_dates <- function(x) {
#   # Replace dates before 1900-01-01 with NA
#   x[x < as.Date("1900-01-01")] <- NA_Date_
#   return(x)
# }

# if (is_unix) {
#   result[, (date_cols) := mclapply(.SD, convert_and_filter_dates, mc.cores = parallel::detectCores()), .SDcols = date_cols]
# } else {
#   result[, (date_cols) := lapply(.SD, convert_and_filter_dates), .SDcols = date_cols]
# }


In [139]:
date_cols <- c("date_adm", "date_dis", "date_rec", "date_ref", "date_check", "pat_bdate", "date_ext")

# Find rows where any date column has a date before 1900-01-01
rows_with_old_dates <- result[Reduce(`|`, lapply(.SD, function(x) x < as.Date("1900-01-01"))), .SDcols = date_cols]

# Print the resulting rows
print(rows_with_old_dates)


Null data.table (0 rows and 0 cols)


In [140]:
# print(head(result, 100))
# Print unique values of the 'clin_discharge' column
# print(unique(result$clin_discharge))


Export to Stata


In [141]:
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")))


# Define a function to convert list columns to character columns
convert_list_to_character <- function(dt) {
  # Loop through all columns in the data.table
  for (col in names(dt)) {
    if (is.list(dt[[col]])) {
      dt[, (col) := sapply(dt[[col]], function(x) {
        if (is.null(x) || all(is.na(x))) {
          return(NA_character_) # Return NA if the list is empty or all values are NA
        } else {
          return(paste(sort(x), collapse = ",")) # Sort and concatenate
        }
      })]
    }
  }
}

# Apply the conversion function to your data.table 'result'
convert_list_to_character(result)

# Export to Stata (.dta) format
write_dta(as.data.frame(result), here(checkpoint_10_path, paste0(checkpoint_10_prefix, year_to_load, suffix, ".dta")))

gcs_auth(email = gcs_email)
gcs_upload(
  file = here(checkpoint_10_path, paste0(checkpoint_10_prefix, year_to_load, suffix, ".dta")),
  bucket = gcs_bucket,
  name = paste0(gcs_spc_fpath, "/", paste0(checkpoint_10_prefix, year_to_load, suffix, ".dta")),
  predefinedAcl = "bucketLevel"
)


ℹ 2024-10-15 09:12:59.405274 > File size detected as  3.9 Mb



==Google Cloud Storage Object==
Name:                spc/stata2022_sampled_1361_.dta 
Type:                application/octet-stream 
Size:                3.9 Mb 
Media URL:           https://www.googleapis.com/download/storage/v1/b/phic-claims-checkpoints/o/spc%2Fstata2022_sampled_1361_.dta?generation=1728983579566621&alt=media 
Download URL:        https://storage.cloud.google.com/phic-claims-checkpoints/spc%2Fstata2022_sampled_1361_.dta 
Public Download URL: https://storage.googleapis.com/phic-claims-checkpoints/spc%2Fstata2022_sampled_1361_.dta 
Bucket:              phic-claims-checkpoints 
ID:                  phic-claims-checkpoints/spc/stata2022_sampled_1361_.dta/1728983579566621 
MD5 Hash:            wz+MmeEqrtBzA9dHHFb6PQ== 
Class:               STANDARD 
Created:             2024-10-15 09:12:59 
Updated:             2024-10-15 09:12:59 
Generation:          1728983579566621 
Meta Generation:     1 
eTag:                CJ3E1OCFkIkDEAE= 
crc32c:              PPk+wg== 

### Push to BQ

Push results to BQ if appropriate (i.e. if allowed by parameters)

In [142]:
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))
result[, c("c1", "c2", "clin_rvs") := NULL]
# cat(unique(result$id_series), sep = "\n")


In [143]:
# test <- fread("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/claims/raw/claims_extract_CLAIMS 2022.tsv", colClasses = "character", nrows = 100)

# print(head(test, 100))


In [144]:
if (nrow(result) == total_rows) bq_table <- paste0("claims_", year_to_load, "1231")

# Check if the table should be dropped and replaced
if (to_drop_bq) {
  tryCatch(
    {
      bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
      message("Table dropped successfully.\n")
    },
    error = function(e) {
      # If the table does not exist, just continue
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.\n")
      } else {
        # If it's a different error, re-throw the error
        stop(e)
      }
    }
  )
}

# Attempt to create the table
tryCatch(
  {
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here("data-cleaning/r_scripts", "bq_schema_cleaning.json"), simplifyDataFrame = FALSE)
    )
    skip_bq_upload <<- FALSE
    message("Table created successfully.\n")
  },
  error = function(e) {
    # Check if the error message indicates that the table already exists
    if (grepl("already exists", e, ignore.case = TRUE)) {
      skip_bq_upload <<- TRUE
      message("Table already exists. Skipping creation and upload.")
    } else {
      # If it's a different error, re-throw the error
      stop(e)
    }
  }
)

# Upload to BQ only if table is empty
if (to_bq && !skip_bq_upload) {
  # tryCatch(
  #   {
  #     bq_table_upload(
  #       bq_table(gcp_proj, bq_dataset, bq_table),
  #       values = result,
  #       write_disposition = "WRITE_EMPTY"
  #     )
  #     message("Data uploaded successfully with WRITE_EMPTY.\n")
  #   },
  #   error = function(e) {
  #     if (grepl("already exists", e, ignore.case = TRUE)) {
  #       # Handle the specific "already exists" error
  #       message("Upload skipped: table already exists and is not empty.")
  #     } else {
  #       # Handle all other errors
  #       message("Error during upload: ", e)
  #     }
  #   }
  # )
  chunk_size <- 1000000 # Adjust the chunk size based on memory availability
  num_chunks <- ceiling(nrow(result) / chunk_size)

  for (i in seq_len(num_chunks)) {
    chunk <- result[((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)), ]

    bq_table_upload(
      bq_table(gcp_proj, bq_dataset, bq_table),
      values = chunk,
      write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
    )
  }
  # gcs_auth(email = gcs_email)
  # gcs_upload(
  #   file = here(checkpoint_4_path, paste0(checkpoint_4_prefix, year_to_load, suffix, ".txt")),
  #   bucket = gcs_bucket,
  #   name = paste0(gcs_pre_fpath, "/", paste0(checkpoint_4_prefix, year_to_load, suffix, ".txt")),
  #   predefinedAcl = "bucketLevel"
  # )
}


Table dropped successfully.




Table created successfully.




In [145]:
# str(result)


## Runtime Estimation

In [146]:
# Print time estimates along with estimate for full claims file
print_time_estimates()


Time spent (total)               : 69.691 sec elapsed
Time spent (t/row) for 20.4k rows: 3.41 msec
Time (est) (total) for 12.8m rows: 725.95 min


## Debugging

In [147]:
# in case we want to run this cell independently:
source(here::here("data-cleaning/r_scripts", "03_timing-debug-functions.R"))

# Consolidate all r_scripts scripts into debug.R; useful for debugging
concatenate_r_files(
  here::here("data-cleaning/r_scripts"),
  here::here("data-cleaning/debug/debug.R")
)

if (.Platform$OS.type == "unix") system("cd ~/drg-pipeline && jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb --output debug/drg-cleaning")


## Cleanup

In [148]:
if (FALSE) {
  to_flush_master <- to_debug <- TRUE
  to_flush_partial <- FALSE
}

# Define the paths and corresponding conditions
paths <- list(
  to_flush_master = c(
    "data-cleaning/cache",
    "data-cleaning/data/profvis",
    "data-cleaning/data/aux-files",
    "data-cleaning/data/checkpoints",
    "data-cleaning/debug"
  ),
  to_flush_partial = c(
    "data-cleaning/data/claims/raw/parts",
    "data-cleaning/data/claims/raw/samples"
  )
)

# Iterate over the paths and conditions to delete them if the condition is true
for (condition in names(paths)) {
  if (get(condition)) {
    system(paste(
      "rm -r",
      paste(here::here(unlist(paths[[condition]])), collapse = " ")
    ))
  }
}

if (to_debug) {
  rm(list = ls())
  gc()
}
